# 01 — Förbered Mustache-datasetet
Samplar mustasch/clean-bilder från CelebA (inkl. kvinnor) och kopierar till `data/dataset_v3/`.
Körs en gång (eller om du vill bygga ett helt nytt dataset från scratch).

In [ ]:
import pandas as pd
import os
import shutil

labels     = pd.read_csv('data/list_attr_celeba.csv')
source_dir = 'data/img/img_align_celeba'

MUSTACHE_DIR = 'data/dataset_v3/mustache'
CLEAN_DIR    = 'data/dataset_v3/clean'

print('Redo!')

## Filtrera och sampla
- Mustasch: alla män med `Mustache == 1`
- Clean: män utan mustasch + kvinnor (extra vikt på svarthåriga kvinnor för bättre representation av hudton/kontrast)

In [ ]:
men   = labels[labels['Male'] ==  1]
women = labels[labels['Male'] == -1]

mustache_df = men[men['Mustache'] == 1]
men_clean   = men[men['Mustache'] == -1]

women_black_hair = women[
    (women['Mustache'] == -1) &
    (women['Black_Hair'] == 1)
].sample(n=2000, random_state=42)

women_other = women[
    (women['Mustache'] == -1) &
    (women['Black_Hair'] == -1)
].sample(n=1000, random_state=42)

women_clean = pd.concat([women_black_hair, women_other])
clean_df    = pd.concat([men_clean, women_clean])

# 5000 per klass — städas i 03_clean_mustache_dataset.ipynb
mustache_sample = mustache_df.sample(n=5000, random_state=42)
clean_sample    = clean_df.sample(n=5000, random_state=42)

print(f'Mustasch: {len(mustache_sample)}')
print(f'Clean:    {len(clean_sample)}')

## Kopiera till dataset_v3

In [ ]:
if os.path.exists('data/dataset_v4'):
    shutil.rmtree('data/dataset_v4')
os.makedirs(MUSTACHE_DIR)
os.makedirs(CLEAN_DIR)

for filename in mustache_sample['image_id']:
    src = os.path.join(source_dir, filename)
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(MUSTACHE_DIR, filename))

for filename in clean_sample['image_id']:
    src = os.path.join(source_dir, filename)
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(CLEAN_DIR, filename))

print(f'Mustasch: {len(os.listdir(MUSTACHE_DIR))}')
print(f'Clean:    {len(os.listdir(CLEAN_DIR))}')

## Klart!
Gå vidare till **02_train_mustache_model.ipynb**.